# Планирование последовательностей интенций поверх FB — воспроизведение

Итог уже известен и получен на CPU: **метод проигрывает бейзлайну** (0.730
против 0.797, парная разность −0.067 с CI [−0.080, −0.050]). Подробности в
`REPORT.md`.

Смысл этого ноутбука — не пересчитать то же самое быстрее, а закрыть
**единственный открытый вопрос**, который CPU не потянул.

Диагноз из отчёта: узкое место — качество попарных оценок достижимости.
Корреляция стоимости с истинным расстоянием растёт с размером набора узла:

| членов в наборе | 1 | 8 | 16 | 32 |
|---|---|---|---|---|
| корреляция | 0.30 | 0.455 | 0.505 | 0.528 |

Все числа отчёта получены при **8** членах и 300 узлах — больше на CPU не
помещалось. Вопрос: если дать рёбрам лучшее качество (32 члена, 1000 узлов),
сократится ли разрыв?

Честное ожидание: скорее нет. Даже 0.528 далеко от 0.75, которые даёт прямая
оценка до цели. Но это предсказание, а не замер, и стоит он на GPU минут сорок.


## 1. Установка

In [1]:
!git clone --recursive https://github.com/2FIVE192/fb-multi-intention-planning.git
%cd fb-multi-intention-planning
!pip install -q -r requirements-colab.txt

Cloning into 'fb-multi-intention-planning'...
remote: Enumerating objects: 160, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 160 (delta 73), reused 148 (delta 63), pack-reused 0 (from 0)
Receiving objects: 100% (160/160), 131.93 KiB | 6.00 MiB/s, done.
Resolving deltas: 100% (73/73), done.
Submodule 'third_party/switching-successor-measures' (https://github.com/stestoKTH/switching-successor-measures.git) registered for path 'third_party/switching-successor-measures'
Cloning into '/content/fb-multi-intention-planning/third_party/switching-successor-measures'...
remote: Enumerating objects: 48, done.        
remote: Counting objects: 100% (19/19), done.        
remote: Compressing objects: 100% (9/9), done.        
remote: Total 48 (delta 13), reused 10 (delta 10), pack-reused 29 (from 1)        
Receiving objects: 100% (48/48), 133.94 KiB | 230.00 KiB/s, done.
Resolving deltas: 100% (17/17), done.
Submodule path 'th

In [2]:
import jax
print('устройства jax:', jax.devices())
assert jax.devices()[0].platform == 'gpu', 'GPU не подключён: Среда выполнения -> Сменить среду выполнения'

устройства jax: [CudaDevice(id=0)]


## 2. Данные

In [3]:
!python scripts/download_datasets.py --datasets antmaze-medium-navigate-v0

[get] https://rail.eecs.berkeley.edu/datasets/ogbench/antmaze-medium-navigate-v0.npz
antmaze-medium-navigate-v0.npz: 100% 232M/232M [00:15<00:00, 15.6MB/s]
[ok] /root/.ogbench/data/antmaze-medium-navigate-v0.npz
[get] https://rail.eecs.berkeley.edu/datasets/ogbench/antmaze-medium-navigate-v0-val.npz
antmaze-medium-navigate-v0-val.npz: 100% 23.2M/23.2M [00:02<00:00, 10.2MB/s]
[ok] /root/.ogbench/data/antmaze-medium-navigate-v0-val.npz

готово: /root/.ogbench/data


## 3. Чекпоинты и настройки

In [4]:
!pip -q install gdown
!python -m gdown --folder https://drive.google.com/drive/folders/1dKYhaDJH9lUREo-kUV3AwmTLrxvKO7Ek -O checkpoints

CHECKPOINT = 'checkpoints/medium'
ENV = 'ogbench-antmaze-medium-navigate-v0'

import os
assert os.path.isfile(os.path.join(CHECKPOINT, 'params.pkl')), 'чекпоинт не скачался'
print(sorted(os.listdir(CHECKPOINT)))

Retrieving folder contents
Retrieving folder 1SwRXBwFpsPUNCzC1SoMVKtx540URtJrW giant
Processing file 1ZzEc7ujezMj2U2aUDeeVFp1qUOi1t0Zx flags.json
Processing file 1HOOIW3uQkcwEYEZwLTsXWu-tU7lshkt0 params.pkl
Retrieving folder 1prWCPrmR_TzT9AahjbBrUaLidlycqIWP large
Processing file 19nh-KCENZDYrOa_ME5du4PqNi3BQVlru flags.json
Processing file 1gV70CRaJudM_lkqi3FFYwLTg82DAyodi params.pkl
Retrieving folder 1mg806bd3v28KTm_GUkWQJyPZKvSCOou- medium
Processing file 1r0wR4il8LIbrPKRMakEckCR4MOs7romJ flags.json
Processing file 1683zzDk5v4m2Otad6I2E0TsoYBcGypqB params.pkl
Retrieving folder 1mJZHteZ7mr1WSWuwdoHv5sw-rgO1lcDh teleport
Processing file 1OrHUH4kPwUdQI067sFsq6twMe1oEAE28 flags.json
Processing file 1FCNHFuMlrLJYlNvqJaoaHP2nw0vreWOR params.pkl
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1ZzEc7ujezMj2U2aUDeeVFp1qUOi1t0Zx
To: /content/fb-multi-intention-planning/checkpoints/giant

## 4. Проверки перед прогоном

Тесты логики планирования (чекпоинт не нужен) и калибровка масштабов среды.

In [5]:
!python tests/test_planning.py
!python scripts/calibrate.py

  OK   test_dijkstra_recovers_true_distance
  OK   test_extracted_path_goes_through_the_gap
  OK   test_far_field_hallucination_exists
  OK   test_paired_comparison_and_bootstrap
[graph] ВНИМАНИЕ: граф остался без рёбер, ослабьте max_edge_cost. Планировщик будет работать как бейзлайн.
  OK   test_planner_falls_back_when_graph_is_useless
  OK   test_planner_solves_maze_where_greedy_fails
  OK   test_pruning_respects_threshold

7/7 тестов прошло
Traceback (most recent call last):
  File "/content/fb-multi-intention-planning/scripts/calibrate.py", line 119, in <module>
    main()
  File "/content/fb-multi-intention-planning/scripts/calibrate.py", line 44, in main
    env, train_dataset, _ = make_env_and_datasets(args.env_name, add_info=True)
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/fb-multi-intention-planning/third_party/switching-successor-measures/utils/env_utils.py", line 111, in make_env_and_datasets
    env_and_datasets = ogbenc

## 5. Главный вопрос: помогает ли лучшее качество рёбер

Сравниваем конфигурацию из отчёта (300 узлов, 8 членов) с полной (1000 узлов,
32 члена). Всё остальное совпадает, включая отложенные сиды 1–3 — на них
подбора гиперпараметров не было.

Если разрыв с бейзлайном сократится — диагноз «дело в качестве рёбер» получает
количественное подтверждение и появляется понятное направление работы. Если
нет — значит упирается не в разрешение оценки, а в саму величину.

In [6]:
COMMON = ('--seeds 1,2,3 --num_episodes 20 --replan_every 20 '
          '--execution high --min_commit_steps 40 '
          '--tail_estimate direct --plan_advantage_steps 25')

# А: ровно та конфигурация, которой получены числа отчёта (контроль).
!python scripts/run_eval.py --checkpoint_dir "$CHECKPOINT" --env_name $ENV     --methods baseline,graph {COMMON}     --num_nodes 300 --num_members 8 --member_stride 8 --normalizer_references 1000     --tag gpu_small

# Б: полная конфигурация — рёбра максимального качества.
!python scripts/run_eval.py --checkpoint_dir "$CHECKPOINT" --env_name $ENV     --methods baseline,graph {COMMON}     --num_nodes 1000 --num_members 32 --member_stride 2 --normalizer_references 4000     --tag gpu_full

[exp] среда ogbench-antmaze-medium-navigate-v0
/usr/local/lib/python3.12/dist-packages/glfw/__init__.py:917: GLFWError: (65550) b'X11: The DISPLAY environment variable is missing'
  warnings.warn(message, GLFWError)
/usr/local/lib/python3.12/dist-packages/glfw/__init__.py:917: GLFWError: (65537) b'The GLFW library is not initialized'
  warnings.warn(message, GLFWError)
Traceback (most recent call last):
  File "/content/fb-multi-intention-planning/scripts/run_eval.py", line 190, in <module>
    main()
  File "/content/fb-multi-intention-planning/scripts/run_eval.py", line 131, in main
    exp = Experiment(
          ^^^^^^^^^^^
  File "/content/fb-multi-intention-planning/fbplan/experiment.py", line 78, in __init__
    self.env, self.train_dataset, self.val_dataset = make_env_and_datasets(
                                                     ^^^^^^^^^^^^^^^^^^^^^^
  File "/content/fb-multi-intention-planning/third_party/switching-successor-measures/utils/env_utils.py", line 111, in mak

## 6. Контрольная абляция: глубина плана

Проверка, что главный вывод отчёта воспроизводится и на хороших рёбрах.
Отличие в одном флаге: `dijkstra` — многошаговая композиция, `direct` — план из
одной подцели. На CPU было 0.47 против 0.69.

In [7]:
for tail in ['dijkstra', 'direct']:
    !python scripts/run_eval.py --checkpoint_dir "$CHECKPOINT" --env_name $ENV         --methods graph --seeds 1,2,3 --num_episodes 20 --replan_every 20         --execution high --min_commit_steps 40 --tail_estimate {tail}         --num_nodes 1000 --num_members 32 --member_stride 2 --normalizer_references 4000         --tag gpu_tail_{tail}

[exp] среда ogbench-antmaze-medium-navigate-v0
/usr/local/lib/python3.12/dist-packages/glfw/__init__.py:917: GLFWError: (65550) b'X11: The DISPLAY environment variable is missing'
  warnings.warn(message, GLFWError)
/usr/local/lib/python3.12/dist-packages/glfw/__init__.py:917: GLFWError: (65537) b'The GLFW library is not initialized'
  warnings.warn(message, GLFWError)
Traceback (most recent call last):
  File "/content/fb-multi-intention-planning/scripts/run_eval.py", line 190, in <module>
    main()
  File "/content/fb-multi-intention-planning/scripts/run_eval.py", line 131, in main
    exp = Experiment(
          ^^^^^^^^^^^
  File "/content/fb-multi-intention-planning/fbplan/experiment.py", line 78, in __init__
    self.env, self.train_dataset, self.val_dataset = make_env_and_datasets(
                                                     ^^^^^^^^^^^^^^^^^^^^^^
  File "/content/fb-multi-intention-planning/third_party/switching-successor-measures/utils/env_utils.py", line 111, in mak

## 7. Сводка

In [9]:
import pandas as pd

for tag in ['gpu_small', 'gpu_full', 'gpu_tail_dijkstra', 'gpu_tail_direct']:
    try:
        df = pd.read_csv(f'results/raw/{tag}_episodes.csv')
    except FileNotFoundError:
        continue
    print(f'--- {tag} ---')
    print(df.groupby('method').success.mean().round(3).to_dict())

    if {'graph', 'baseline'} <= set(df.method.unique()):
        import sys; sys.path.insert(0, '.')
        from fbplan.stats import paired_comparison
        cmp = paired_comparison(df, 'graph', 'baseline')
        print(f'  парная разность {cmp["delta"]:+.3f} '
              f'[{cmp["ci_low"]:+.3f}, {cmp["ci_high"]:+.3f}]')
    print()